In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("TMDB Data Cleaning") \
    .getOrCreate()

movies_df = spark.read.parquet("../data/raw/tmdb_movies_raw.parquet")

Dataset dibaca kembali dari folder raw data sehingga proses cleaning dapat dilakukan tanpa mengubah data asli.

# Merubah Tipe Data

In [2]:
movies_df.dtypes

[('Movie Name', 'string'),
 ('Year', 'string'),
 ('Certificate', 'string'),
 ('Duration', 'string'),
 ('Genre', 'string'),
 ('TMDB Rating', 'double'),
 ('Votes', 'bigint'),
 ('Director', 'string'),
 ('Stars', 'array<string>'),
 ('Grossed in $', 'string'),
 ('Plot', 'string'),
 ('Original Language', 'string'),
 ('Popularity', 'double'),
 ('TMDB ID', 'bigint'),
 ('IMDb ID', 'string')]

Langkah ini bertujuan untuk memastikan bahwa setiap atribut dalam dataset memiliki tipe data yang sesuai dengan karakteristik variabelnya.

In [3]:
movies_df = movies_df.withColumn(
    "Year",
    F.col("Year").cast("int")
)

In [4]:
movies_df = movies_df.withColumn(
    "Duration",
    F.regexp_replace("Duration", " min", "")
)

In [5]:
movies_df = movies_df.withColumn(
    "Duration",
    F.col("Duration").cast("int")
)

In [6]:
movies_df = movies_df.withColumn(
    "Grossed in $",
    F.regexp_replace(F.col("Grossed in $"), "[$,]", "")
)

In [7]:
movies_df = movies_df.withColumn(
    "Grossed in $",
    F.col("Grossed in $").cast("double")
)

Transformasi tipe data dilakukan apabila terdapat atribut yang terbaca dengan tipe data yang tidak sesuai.

In [8]:
movies_df.dtypes

[('Movie Name', 'string'),
 ('Year', 'int'),
 ('Certificate', 'string'),
 ('Duration', 'int'),
 ('Genre', 'string'),
 ('TMDB Rating', 'double'),
 ('Votes', 'bigint'),
 ('Director', 'string'),
 ('Stars', 'array<string>'),
 ('Grossed in $', 'double'),
 ('Plot', 'string'),
 ('Original Language', 'string'),
 ('Popularity', 'double'),
 ('TMDB ID', 'bigint'),
 ('IMDb ID', 'string')]

# Mengisi Missing Values

In [9]:
movies_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in movies_df.columns
]).show()

+----------+----+-----------+--------+-----+-----------+-----+--------+-----+------------+----+-----------------+----------+-------+-------+
|Movie Name|Year|Certificate|Duration|Genre|TMDB Rating|Votes|Director|Stars|Grossed in $|Plot|Original Language|Popularity|TMDB ID|IMDb ID|
+----------+----+-----------+--------+-----+-----------+-----+--------+-----+------------+----+-----------------+----------+-------+-------+
|         0|   3|       1312|       3|    0|          0|    0|       1|    0|        2421|   2|                0|         0|      0|      7|
+----------+----+-----------+--------+-----+-----------+-----+--------+-----+------------+----+-----------------+----------+-------+-------+



Analisis missing value dilakukan untuk memahami kualitas data yang akan dianalisis.

In [10]:
movies_df = movies_df.dropna(
    subset=[
        "Year",
        "Duration",
        "Director",
        "Plot",
        "IMDb ID"
    ]
)

Karena data variabel Year, Duration, Director, Plot, dan IMDb ID yang hilang sangat sedikit, saya memilih untuk menghapus saja data kosong dari variabel tersebut.

In [11]:
movies_df = movies_df.fillna({
    "Certificate": "Unknown"
})

Nilai kosong pada kolom Certificate diisi dengan kategori baru yaitu Unknown.

In [12]:
median_gross = movies_df.approxQuantile(
    "Grossed in $",
    [0.5],
    0.01
)[0]

median_gross

28600000.0

In [13]:
movies_df = movies_df.fillna({
    "Grossed in $": median_gross
})

Variabel Grossed in $ merepresentasikan pendapatan film yang bersifat numerik dan memiliki distribusi yang cenderung tidak simetris karena adanya film blockbuster dengan pendapatan sangat tinggi. Oleh karena itu, metode imputasi median dipilih untuk menggantikan nilai yang hilang. Median dianggap lebih robust terhadap nilai ekstrem dibandingkan rata-rata sehingga mampu memberikan estimasi yang lebih representatif terhadap distribusi data.

In [14]:
movies_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in movies_df.columns
]).show()

+----------+----+-----------+--------+-----+-----------+-----+--------+-----+------------+----+-----------------+----------+-------+-------+
|Movie Name|Year|Certificate|Duration|Genre|TMDB Rating|Votes|Director|Stars|Grossed in $|Plot|Original Language|Popularity|TMDB ID|IMDb ID|
+----------+----+-----------+--------+-----+-----------+-----+--------+-----+------------+----+-----------------+----------+-------+-------+
|         0|   0|          0|       0|    0|          0|    0|       0|    0|           0|   0|                0|         0|      0|      0|
+----------+----+-----------+--------+-----+-----------+-----+--------+-----+------------+----+-----------------+----------+-------+-------+



In [15]:
movies_df.count()

9740

Jumlah row dataset setelah dilakukan cleaning ada di 9740 film.

In [16]:
movies_df.write.mode("overwrite").parquet("../data/processed/tmdb_movies_clean.parquet")

Dataset yang telah melalui proses pembersihan disimpan dalam format Parquet pada direktori processed.